## Parte 1: Dimensões (geração de chaves substitutas)

In [0]:
from pyspark.sql import functions as F

torneios = spark.table("workspace.silver.torneios")
decks = spark.table("workspace.silver.decks")
deck_cartas = spark.table("workspace.silver.deck_cartas")
cartas = spark.table("workspace.silver.cartas")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
# DIM_TORNEIO
dim_torneio = torneios.withColumn("torneio_id", F.monotonically_increasing_id()) \
    .select("torneio_id", "torneio_uri", "nome", "data", "fonte")

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_torneio")
dim_torneio.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_torneio")
dim_torneio.count()

In [0]:
# DIM_JOGADOR
dim_jogador = decks.select("nome_jogador").distinct() \
    .withColumn("jogador_id", F.monotonically_increasing_id()) \
    .select("jogador_id", "nome_jogador")

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_jogador")
dim_jogador.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_jogador")
dim_jogador.count()

In [0]:
# DIM_CARTA (já enriquecida com Scryfall na Silver)
dim_carta = cartas.withColumn("carta_id", F.monotonically_increasing_id()) \
    .select("carta_id", "nome_carta", "mana_cost", "cmc", "type_line", "colors")

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_carta")
dim_carta.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_carta")
dim_carta.count()

## Parte 2: DIM_DECK (referenciando as dimensões acima)

In [0]:
decks_com_chaves = decks.join(
    dim_torneio.select("torneio_id", "torneio_uri"), on="torneio_uri", how="left"
).join(
    dim_jogador, on="nome_jogador", how="left"
)

dim_deck = decks_com_chaves.withColumn("deck_id", F.monotonically_increasing_id()) \
    .select("deck_id", "linha_deck", "torneio_id", "jogador_id", "deck_uri")

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_deck")
dim_deck.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_deck")
dim_deck.count()

## Parte 3: Tabelas fato

In [0]:
# FATO_DECK_CARTA
fato_deck_carta = deck_cartas.join(
    dim_deck.select("deck_id", "linha_deck"), on="linha_deck", how="left"
).join(
    dim_carta.select("carta_id", "nome_carta"), on="nome_carta", how="left"
).select("deck_id", "carta_id", "quantidade", "is_sideboard")

spark.sql("DROP TABLE IF EXISTS workspace.gold.fato_deck_carta")
fato_deck_carta.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fato_deck_carta")
fato_deck_carta.count()

In [0]:
# FATO_RESULTADO_DECK
fato_resultado_deck = decks_com_chaves.join(
    dim_deck.select("deck_id", "linha_deck"), on="linha_deck", how="left"
).select("deck_id", "torneio_id", "tipo_resultado", "colocacao", "vitorias", "derrotas")

spark.sql("DROP TABLE IF EXISTS workspace.gold.fato_resultado_deck")
fato_resultado_deck.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fato_resultado_deck")
fato_resultado_deck.count()